# Pyfolio Style Evaluation

Modernized, dependency-light version for Python 3.10+.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 80)
FOLDER = Path.cwd().resolve()
if str(FOLDER) not in sys.path:
    sys.path.insert(0, str(FOLDER))
from strategy_evaluation_utils import (
    make_price_panel, returns_matrix, moving_average_signal, equal_weight_returns,
    performance_table, mean_variance_weights, portfolio_returns, kelly_fraction, export_strategy_outputs
)
OUTPUT = FOLDER.parent / 'data' / 'strategy_evaluation'
OUTPUT.mkdir(parents=True, exist_ok=True)
prices = make_price_panel()
rets = returns_matrix(prices)


In [2]:
signals = moving_average_signal(prices)
strategy = signals.groupby('date')['strategy_return'].mean().rename('ma_strategy')
benchmark = equal_weight_returns(rets)
comparison = pd.concat([strategy, benchmark], axis=1).dropna()
comparison['active_return'] = comparison['ma_strategy'] - comparison['equal_weight']
comparison['cum_strategy'] = (1 + comparison['ma_strategy']).cumprod()
comparison['cum_benchmark'] = (1 + comparison['equal_weight']).cumprod()
comparison['drawdown'] = comparison['cum_strategy'] / comparison['cum_strategy'].cummax() - 1
summary = performance_table({'ma_strategy': comparison['ma_strategy'], 'benchmark': comparison['equal_weight'], 'active': comparison['active_return']})
export_strategy_outputs(OUTPUT, pyfolio_style_returns=comparison, pyfolio_style_summary=summary)
display(summary)
display(comparison.tail())

,strategy,annual_return,annual_volatility,sharpe,max_drawdown,hit_rate
1,benchmark,0.137770,0.095225,1.446789,-0.088848,0.519841
0,ma_strategy,0.101225,0.073000,1.386637,-0.068050,0.515873
2,active,-0.032136,0.062710,-0.512446,-0.175782,0.486772


,ma_strategy,equal_weight,active_return,cum_strategy,cum_benchmark,drawdown
date,,,,,,
2023-11-21,-0.002982,0.005652,-0.008634,1.315965,1.458588,-0.021083
2023-11-22,0.002240,0.004699,-0.002459,1.318914,1.465442,-0.018890
2023-11-23,0.001920,0.000406,0.001514,1.321446,1.466038,-0.017005
2023-11-24,-0.000610,0.004364,-0.004974,1.320640,1.472435,-0.017605
2023-11-27,0.003184,-0.013186,0.016370,1.324844,1.453020,-0.014478
